# Sharing Models and Tokenizers

## The Hugging Face Hub

Hugging face hub is a central platform that enables anyone to discover, use, and contribute new state-of-the-art models and datasets. It hosts a wide variety of models, with more than 10,000 publicly available.

The models in the Hub are not limited to 🤗 Transformers or even NLP. There are models from Flair and AllenNLP for NLP, Asteroid and pyannote for speech, and timm for vision, to name a few.

Each of these models is hosted as a Git repository, which allows versioning and reproducibility. Sharing a model on the Hub means opening it up to the community and making it accessible to anyone looking to easily use it, in turn eliminating their need to train a model on their own and simplifying sharing and usage.

Additionally, sharing a model on the Hub automatically deploys a hosted Inference API for that model.

##  Using pretrained models

The Model Hub makes selecting the appropriate model simple, so that using it in any downstream library can be done in a few lines of code. Let’s take a look at how to actually use one of these models, and how to contribute back to the community.

Let’s say we’re looking for a French-based model that can perform mask filling.

We select the camembert-base checkpoint to try it out. The identifier camembert-base is all we need to start using it!

In [8]:
from transformers import pipeline

camembert_fill_mask = pipeline("fill-mask", model="camembert-base")
results = camembert_fill_mask("Le camembert est <mask>")
results

[{'score': 0.32866090536117554,
  'token': 23,
  'token_str': 'un',
  'sequence': 'Le camembert est un'},
 {'score': 0.18298353254795074,
  'token': 186,
  'token_str': '...',
  'sequence': 'Le camembert est...'},
 {'score': 0.11833496391773224,
  'token': 28,
  'token_str': 'une',
  'sequence': 'Le camembert est une'},
 {'score': 0.10589142888784409,
  'token': 807,
  'token_str': '[...]',
  'sequence': 'Le camembert est [...]'},
 {'score': 0.046711575239896774,
  'token': 16,
  'token_str': 'le',
  'sequence': 'Le camembert est le'}]

As you can see, loading a model within a pipeline is extremely simple. The only thing you need to watch out for is that the chosen checkpoint is suitable for the task it’s going to be used for. For example, here we are loading the camembert-base checkpoint in the fill-mask pipeline, which is completely fine. But if we were to load this checkpoint in the text-classification pipeline, the results would not make any sense because the head of camembert-base is not suitable for this task! 

we can also instantiate the checkpoint using the model architecture directly:

In [9]:
from transformers import BertTokenizer, BertForMaskedLM

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

Downloading:   0%|          | 0.00/232k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/570 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


While the previous code sample limits users to checkpoints loadable in the BERT architecture, using the Auto* classes makes switching checkpoints simple:

In [10]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = AutoModelForMaskedLM.from_pretrained("camembert-base")

###  Sharing pretrained models

we’ll take a look at the easiest ways to share pretrained models to the 🤗 Hub. There are tools and utilities available that make it simple to share and update models directly on the Hub.

There are three ways to go about creating new model repositories:

    Using the push_to_hub API
    Using the huggingface_hub Python library
    Using the web interface

Once you’ve created a repository, you can upload files to it via git and git-lfs.

###  Using the push_to_hub API

The simplest way to upload files to the Hub is by leveraging the push_to_hub API.

Before going further, you’ll need to generate an authentication token so that the huggingface_hub API knows who you are and what namespaces you have write access to

from huggingface_hub import notebook_login


```

notebook_login()

# In a terminal, you can run:

huggingface-cli login
```



In both cases, you should be prompted for your username and password, which are the same ones you use to log in to the Hub.

The easiest way to upload it to the Hub is to set push_to_hub=True when you define your TrainingArguments:

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    "bert-finetuned-mrpc", save_strategy="epoch", push_to_hub=True
)

When you call trainer.train(), the Trainer will then upload your model to the Hub each time it is saved (here every epoch) in a repository in your namespace. That repository will be named like the output directory you picked (here bert-finetuned-mrpc).


To upload a model to an organization you are a member of, just pass it with hub_model_id = "my_organization/my_repo_name".

Once your training is finished, you should do a final trainer.push_to_hub() to upload the last version of your model. It will also generate a model card with all the relevant metadata, reporting the hyperparameters used and the evaluation results!

At a lower level, accessing the Model Hub can be done directly on models, tokenizers, and configuration objects via their push_to_hub() method. This method takes care of both the repository creation and pushing the model and tokenizer files directly to the repository. No manual handling is required, unlike with the API we’ll see below.

To get an idea of how it works, let’s first initialize a model and a tokenizer:

In [12]:
from transformers import AutoModelForMaskedLM, AutoTokenizer

checkpoint = "camembert-base"

model = AutoModelForMaskedLM.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

second step:

In [ ]:
model.push_to_hub("repo-name")

This will create the new repository repo-name in your profile, and populate it with your model files. Do the same with the tokenizer, so that all the files are now available in this repository:

In [ ]:
tokenizer.push_to_hub("repo-name")

If you belong to an organization, simply specify the organization argument to upload to that organization’s namespace:

In [ ]:
tokenizer.push_to_hub("repo-name", organization="niffoxic", use_auth_token="<TOKEN>")

the push_to_hub() method accepts several arguments, making it possible to upload to a specific repository or organization namespace, or to use a different API token. 

The push_to_hub() method is backed by the huggingface_hub Python package, which offers a direct API to the Hugging Face Hub. It’s integrated within 🤗 Transformers and several other machine learning libraries, like allenlp.

### Using the huggingface_hub Python library

The huggingface_hub Python library is a package which offers a set of tools for the model and datasets hubs. It provides simple methods and classes for common tasks like getting information about repositories on the hub and managing them. It provides simple APIs that work on top of git to manage those repositories’ content and to integrate the Hub in your projects and libraries.

Similarly to using the push_to_hub API, this will require you to have your API token saved in your cache. In order to do this, you will need to use the login command from the CLI



```
huggingface-cli login
```

The huggingface_hub package offers several methods and classes which are useful for our purpose. Firstly, there are a few methods to manage repository creation, deletion, and others:




```

from huggingface_hub import (
    # User management
    login,
    logout,
    whoami,

    # Repository creation and management
    create_repo,
    delete_repo,
    update_repo_visibility,

    # And some methods to retrieve/change information about the content
    list_models,
    list_datasets,
    list_metrics,
    list_repo_files,
    upload_file,
    delete_file,
)

```



Additionally, it offers the very powerful Repository class to manage a local repository.

The create_repo method can be used to create a new repository on the hub:

In [ ]:
from huggingface_hub import create_repo

create_repo("repo-name")

This will create the repository repo-name in your namespace. If you like, you can specify which organization the repository should belong to using the organization argument:

In [ ]:
from huggingface_hub import create_repo

create_repo("dummy-model", organization="huggingface")

This will create the dummy-model repository in the huggingface namespace, assuming you belong to that organization. Other arguments which may be useful are:

    private, in order to specify if the repository should be visible from others or not.
    token, if you would like to override the token stored in your cache by a given token.
    repo_type, if you would like to create a dataset or a space instead of a model. Accepted values are "dataset" and "space".

Once the repository is created, we should add files to it! 

###  Using the web interface

The web interface offers tools to manage repositories directly in the Hub. Using the interface, you can easily create repositories, add files (even large ones!), 


First, specify the owner of the repository: this can be either you or any of the organizations you’re affiliated with. If you choose an organization, the model will be featured on the organization’s page and every member of the organization will have the ability to contribute to the repository.

Next, enter your model’s name. This will also be the name of the repository. Finally, you can specify whether you want your model to be public or private. Private models are hidden from public view.

This is where your model will be hosted. To start populating it, you can add a README file directly from the web interface.

The README file is in Markdown — feel free to go wild with it! The third part of this chapter is dedicated to building a model card. These are of prime importance in bringing value to your model, as they’re where you tell others what it can do.

If you look at the “Files and versions” tab, you’ll see that there aren’t many files there yet — just the README.md you just created and the .gitattributes file that keeps track of large files.



###  Uploading the model files

The system to manage files on the Hugging Face Hub is based on git for regular files, and git-lfs (which stands for Git Large File Storage) for larger files.

###  The upload_file approach

Using upload_file does not require git and git-lfs to be installed on your system. It pushes files directly to the 🤗 Hub using HTTP POST requests. A limitation of this approach is that it doesn’t handle files that are larger than 5GB in size. If your files are larger than 5GB.

In [ ]:
from huggingface_hub import upload_file

upload_file(
    "<path_to_file>/config.json",
    path_in_repo="config.json",
    repo_id="<namespace>/dummy-model",
)

This will upload the file config.json available at <path_to_file> to the root of the repository as config.json, to the dummy-model repository. Other arguments which may be useful are:

    token, if you would like to override the token stored in your cache by a given token.
    repo_type, if you would like to upload to a dataset or a space instead of a model. Accepted values are "dataset" and "space".

###  The Repository class

The Repository class manages a local repository in a git-like manner. It abstracts most of the pain points one may have with git to provide all features that we require.

Using this class requires having git and git-lfs installed.


In order to start playing around with the repository we have just created, we can start by initialising it into a local folder by cloning the remote repository:


In [ ]:
from huggingface_hub import Repository

repo = Repository("<path_to_dummy_folder>", clone_from="<namespace>/dummy-model")

This created the folder <path_to_dummy_folder> in our working directory. This folder only contains the .gitattributes file as that’s the only file created when instantiating the repository through create_repo.

From this point on, we may leverage several of the traditional git methods:



```
repo.git_pull()
repo.git_add()
repo.git_commit()
repo.git_push()
repo.git_tag()
```



we have a model and a tokenizer that we would like to push to the hub. We have successfully cloned the repository, we can therefore save the files within that repository.

We first make sure that our local clone is up to date by pulling the latest changes:



```
repo.git_pull()
```

Once that is done, we save the model and tokenizer files:



```
model.save_pretrained("<path_to_dummy_folder>")
tokenizer.save_pretrained("<path_to_dummy_folder>")
```


The <path_to_dummy_folder> now contains all the model and tokenizer files. We follow the usual git workflow by adding files to the staging area, committing them and pushing them to the hub:



```
repo.git_add()
repo.git_commit("Add model and tokenizer files")
repo.git_push()
```


### The git-based approach


This is the very barebones approach to uploading files: we’ll do so with git and git-lfs directly. Most of the difficulty is abstracted away by previous approaches, but there are a few caveats with the following method so we’ll follow a more complex use-case.



```
git lfs install
```

Once that’s done, the first step is to clone your model repository:


```
git clone https://huggingface.co/<namespace>/<your-model-id>
```

I’ve used the model name dummy, so for me the command ends up looking like the following:


```
git clone https://huggingface.co/niffoxic/dummy
```

I now have a folder named dummy in my working directory. I can cd into the folder and have a look at the contents:



```
cd dummy && ls
```


If you just created your repository using Hugging Face Hub’s create_repo method, this folder should only contain a hidden .gitattributes file.

# Thank you if anybody is reading this, I just wanna say how good hugging face and pytorch and fast ai and deep learning is!! bro I'm in love with this sh*t <3